In [43]:
import torch
from torch import nn
from torch.nn import functional as F

In [44]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = 'cpu'
print(device)

cuda


In [64]:
with open('input.txt' , 'r' , encoding = 'UTF-8') as f:
    text = f.read()
torch.manual_seed(1337)

In [46]:
vocab = sorted(list(set(text)))

stoi = {ch : i for i , ch in enumerate(vocab)}
itos = {i : ch for i , ch in enumerate(vocab)}
vocab_size = len(vocab)


def encode(s):
    return [stoi[c] for c in s]
def decode(n):
    return ''.join([itos[i] for i in n])


In [47]:
embd_dim = 384
n_head = 6
block_size = 256
batch_size = 64
head_size = int(embd_dim / n_head)
n_layers = 6
eval_iters = 200
max_iters = 5000
eval_interval = 500

In [68]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [66]:
class Head(nn.Module):
    def __init__(self , head_size):
        super().__init__()

        dropout = 0.2

        self.key = nn.Linear(embd_dim  , head_size , bias = False)
        self.query = nn.Linear(embd_dim  , head_size , bias = False)
        self.value = nn.Linear(embd_dim  , head_size , bias = False)

        self.register_buffer('tril' , torch.tril(torch.ones(block_size , block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self , x):

        B , T , C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * head_size**-0.5
        wei = wei.masked_fill(self.tril[:T , :T] == 0 , float('-inf'))
        wei = F.softmax(wei , dim = -1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

In [50]:
class MultiHeadAttention(nn.Module):
    def __init__(self , n_head):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])

        self.proj = nn.Linear(embd_dim , embd_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self , x):
        out = torch.cat([h(x) for h in self.heads] , dim = -1)
        out = self.dropout(self.proj(out))
        return out

In [59]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(0.2),
        )

    def forward(self, x):
        return self.net(x)

In [60]:
class Block(nn.Module):
    def __init__(self, n_head, embd_dim):
        super().__init__()

        self.sa = MultiHeadAttention(n_head)
        self.ffwd = FeedForward(embd_dim)
        self.ln1 = nn.LayerNorm(embd_dim)
        self.ln2 = nn.LayerNorm(embd_dim)

    def forward(self , x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [67]:
class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size , embd_dim)
        self.position_embedding_table = nn.Embedding(block_size , embd_dim)
        self.blocks = nn.Sequential(*[Block(n_head , embd_dim) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embd_dim)
        self.lm_head = nn.Linear(embd_dim , vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self , module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx , targets = None):
        B,T = idx.shape
        token_embd = self.token_embedding_table(idx)
        pos_embd = self.position_embedding_table(torch.arange(T, device=device))
        x = token_embd + pos_embd
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self , idx , max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[: , -block_size:]
            logits , loss = self(idx_cond)
            logits = logits[:, -1, :]
            prob = F.softmax(logits , dim = -1)
            idx_next = torch.multinomial(prob , num_samples = 1)
            idx = torch.cat((idx , idx_next) , dim = 1)
        return idx

In [69]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
model = GPTModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4 , weight_decay=1e-4)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

10.788929 M parameters
step 0: train loss 4.2221, val loss 4.2306
step 500: train loss 1.7583, val loss 1.9146
step 1000: train loss 1.3946, val loss 1.6054
step 1500: train loss 1.2670, val loss 1.5291
step 2000: train loss 1.1861, val loss 1.4979
